# 01. Web Crawling Berita Detik.com (Sport & Finance)

Notebook ini berisi proses **Web Crawling / Scraping** untuk mengumpulkan artikel berita berbahasa Indonesia dari portal berita **Detik.com** secara langsung.

### Target Dataset:
- **100 berita Sport** dari `https://sport.detik.com/indeks`
- **100 berita Finance** dari `https://finance.detik.com/indeks`
- Output disimpan ke file: `dataset_detik.csv`

## 1. Install & Import Library
Library `requests` untuk HTTP request, `beautifulsoup4` untuk parsing HTML, dan `pandas` untuk menyimpan data ke DataFrame.

In [1]:
%pip install -q requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Fungsi Scraping Berita Detik
Fungsi `crawl_detik` mengumpulkan URL dari halaman indeks dan mengekstrak judul serta paragraf isi artikel.

In [2]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def crawl_kategori(base_url, label, prefix, target=100):
    data = []
    urls = set()
    page = 1
    
    print(f"--- Mulai crawling {label.upper()} (Target: {target}) ---")
    while len(data) < target and page <= 25:
        page_url = f"{base_url}?page={page}"
        try:
            res = requests.get(page_url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')
            
            # Cari link artikel berita
            for a in soup.find_all('a', href=True):
                href = a['href'].split('?')[0]
                if '/d-' in href and not any(x in href for x in ['/foto', '/tag', '/video', '/detiktv', '/ad']):
                    if href not in urls:
                        urls.add(href)
                        
                        # Ambil isi artikel
                        try:
                            art_res = requests.get(href, headers=HEADERS, timeout=8)
                            art_soup = BeautifulSoup(art_res.text, 'html.parser')
                            
                            # Judul & teks
                            h1 = art_soup.find('h1')
                            body = art_soup.find('div', class_='detail__body-text') or art_soup.find('div', class_='itp_bodycontent')
                            if h1 and body:
                                paragraphs = [p.get_text(strip=True) for p in body.find_all('p') if p.get_text(strip=True)]
                                content = h1.get_text(strip=True) + ". " + " ".join(paragraphs)
                                content = re.sub(r'\s+', ' ', content).strip()
                                
                                if len(content) > 150:
                                    nomor = len(data) + 1
                                    data.append({
                                        'id': f"{prefix}{nomor:03d}",
                                        'isi_berita': content,
                                        'label': label,
                                        'url': href
                                    })
                                    if len(data) % 20 == 0 or len(data) == target:
                                        print(f"Terkumpul {len(data)}/{target} berita {label}")
                                    if len(data) >= target:
                                        break
                        except:
                            pass
        except Exception as e:
            print(f"Error halaman {page}: {e}")
        page += 1
        time.sleep(0.1)
        
    return data

## 3. Eksekusi Crawling 100 Sport & 100 Finance

In [ ]:
# Jalankan crawling untuk sport dan finance
data_sport = crawl_kategori('https://sport.detik.com/indeks', 'sport', 'SP', target=100)
data_finance = crawl_kategori('https://finance.detik.com/indeks', 'finance', 'FN', target=100)

# Gabungkan hasil crawling
df_crawled = pd.DataFrame(data_sport + data_finance)
print("\nTotal data terkumpul:", len(df_crawled))

--- Mulai crawling SPORT (Target: 100) ---


## 4. Validasi & Penyimpanan Dataset
Memastikan tidak ada URL atau isi berita yang duplikat, tidak ada nilai kosong, lalu menyimpannya ke `dataset_detik.csv`.

In [ ]:
# Validasi data
print("Distribusi Label:\n", df_crawled['label'].value_counts())
print("Jumlah Nilai Kosong:\n", df_crawled.isnull().sum())
print("Duplikat URL:", df_crawled['url'].duplicated().sum())
print("Duplikat Isi Berita:", df_crawled['isi_berita'].duplicated().sum())

# Simpan ke dataset_detik.csv jika sudah mencapai target 200 data
if len(df_crawled) == 200:
    df_crawled.to_csv('dataset_detik.csv', index=False)
    print("\nDataset berhasil disimpan ke 'dataset_detik.csv'.")
else:
    print("\nJumlah data belum 200, silakan sesuaikan batas halaman crawling.")

display(df_crawled.head())